# Bank Log-Loss & Cost – Solution

**Short name (GitHub):** `Bank_LogLoss_Cost`  
Work the skeleton first. This notebook is the worked key for the retail-credit adaptation of C1_W3 Lab04 + Lab05.


## Cheat-sheet
PD $f=g(w\cdot x+b)$. Book cost $J=\mathrm{mean}(L)$. Lab05 check: $J(b=-3)\approx0.36687$, $J(b=-4)\approx0.50368$. ECL $=\mathrm{PD}\times\mathrm{LGD}\times\mathrm{EAD}$ is a different number.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=6, suppress=True)



In [ ]:
arr = np.loadtxt("data/bank_dti_1d.csv", delimiter=",", skiprows=1)
dti, y_def = arr[:, 0], arr[:, 1]
print("n loans", dti.shape[0], "default rate", y_def.mean())

fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.scatter(dti[y_def == 0], y_def[y_def == 0], c="#1f77b4", s=70, label="paid")
ax.scatter(dti[y_def == 1], y_def[y_def == 1], c="#d62728", marker="x", s=80, label="defaulted")
ax.set_xlabel("DTI"); ax.set_ylabel("outcome"); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title("DTI book")
plt.show()



In [ ]:
def sigmoid(z):
    z = np.clip(np.asarray(z, dtype=float), -50, 50)
    return 1.0 / (1.0 + np.exp(-z))

def squared_pd_cost(x, y, w, b):
    return float(np.mean((sigmoid(w * x + b) - y) ** 2))

print("sigmoid(0) =", sigmoid(0.0))
print("J_sq tight =", squared_pd_cost(dti, y_def, 18.0, -6.8))
print("J_sq loose =", squared_pd_cost(dti, y_def, 4.0, -1.0))



**Alternate:** pure Python loop for squared PD error — same value, easier to trace one loan.


In [ ]:
def squared_pd_cost_loop(x, y, w, b):
    acc = 0.0
    for i in range(len(x)):
        acc += (float(sigmoid(w * x[i] + b)) - y[i]) ** 2
    return acc / len(x)
print("loop J_sq tight =", squared_pd_cost_loop(dti, y_def, 18.0, -6.8))



In [ ]:
def _clip(f):
    return np.clip(np.asarray(f, dtype=float), 1e-15, 1 - 1e-15)

def loan_loss_piecewise(f, y):
    f = float(_clip(f)); y = float(y)
    return -np.log(f) if y == 1 else -np.log(1 - f)

def loan_loss_compact(f, y):
    f = _clip(f); y = np.asarray(y, dtype=float)
    return -(y * np.log(f) + (1 - y) * np.log(1 - f))

for f, y in [(0.20, 1), (0.05, 0), (0.85, 0), (0.90, 1)]:
    a = loan_loss_piecewise(f, y)
    b = float(loan_loss_compact(np.array([f]), np.array([y])))
    print(f"PD={f}, defaulted={int(y)}  piecewise={a:.4f}  compact={b:.4f}")



In [ ]:
f = np.linspace(1e-3, 1-1e-3, 400)
fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.plot(f, -np.log(f), label="defaulted: −log(PD)")
ax.plot(f, -np.log(1-f), label="paid: −log(1−PD)")
ax.set_xlabel("predicted PD"); ax.set_ylabel("per-loan loss")
ax.set_title("Confident wrong PDs are expensive")
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()



**Alternate:** `np.where` for an already-vector of outcomes.


In [ ]:
def loan_loss_where(f, y):
    f = _clip(f); y = np.asarray(y, dtype=float)
    return np.where(y == 1, -np.log(f), -np.log(1 - f))
print(loan_loss_where([0.2, 0.85], [1, 0]))



In [ ]:
def compute_cost_logistic(X, y, w, b):
    X = np.asarray(X, dtype=float); y = np.asarray(y, dtype=float)
    m = X.shape[0]; acc = 0.0
    for i in range(m):
        z = (w * X[i] + b) if X.ndim == 1 else (np.dot(X[i], w) + b)
        acc += float(loan_loss_compact(sigmoid(z), y[i]))
    return acc / m

def compute_cost_logistic_vec(X, y, w, b):
    X = np.asarray(X, dtype=float); y = np.asarray(y, dtype=float); w = np.asarray(w, dtype=float)
    z = (w * X + b) if X.ndim == 1 else (X @ w + b)
    return float(np.mean(loan_loss_compact(sigmoid(z), y)))

print("1-D J tight =", compute_cost_logistic(dti, y_def, 18.0, -6.8))
print("1-D J loose =", compute_cost_logistic(dti, y_def, 4.0, -1.0))
print("vec matches", np.isclose(compute_cost_logistic(dti, y_def, 18.0, -6.8),
                               compute_cost_logistic_vec(dti, y_def, 18.0, -6.8)))



**Alternate #3:** `sklearn.metrics.log_loss(y, pd)` if sklearn is installed — same mean BCE with internal clipping.


In [ ]:
raw = np.loadtxt("data/bank_scorecard_2d.csv", delimiter=",", skiprows=1)
X2, y2 = raw[:, :2], raw[:, 2]
w = np.array([1.0, 1.0])
print("J S1 b=-3 =", compute_cost_logistic(X2, y2, w, -3))
print("J S2 b=-4 =", compute_cost_logistic(X2, y2, w, -4))

x0 = np.linspace(0, 4, 50)
fig, ax = plt.subplots(figsize=(5.1, 4.3))
ax.scatter(X2[y2 == 0, 0], X2[y2 == 0, 1], label="paid")
ax.scatter(X2[y2 == 1, 0], X2[y2 == 1, 1], marker="x", label="defaulted")
ax.plot(x0, 3 - x0, label="S1 b=-3")
ax.plot(x0, 4 - x0, ls="--", label="S2 b=-4")
ax.set_xlim(0, 4); ax.set_ylim(0, 3.6)
ax.set_xlabel("utilisation index"); ax.set_ylabel("DTI index")
ax.set_title("S1 is the cheaper scorecard")
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()



In [ ]:
f_tab = np.array([0.12, 0.08, 0.78])
y_tab = np.array([1.0, 0.0, 0.0])
L = loan_loss_compact(f_tab, y_tab)
print("losses", L, "mean J", float(np.mean(L)))
print("L3 is the expensive file: paid, but PD=0.78")



In [ ]:
book = np.loadtxt("data/bank_book_practice.csv", delimiter=",", skiprows=1)
Xp = np.column_stack([book[:, 0] - 660.0, book[:, 1] - 0.32])
yp = book[:, 2]
print("book model J =", compute_cost_logistic_vec(Xp, yp, np.array([-0.012, 8.5]), 0.0))
print("naive 50/50 J =", compute_cost_logistic_vec(Xp, yp, np.array([0.0, 0.0]), 0.0))
print("−log(0.5)     =", -np.log(0.5))



In [ ]:
ead, lgd = 20000.0, 0.45
for pd in (0.10, 0.80):
    print(f"PD={pd:.2f}  logistic loss (y=1)={-np.log(pd):.3f}  ECL=${pd*lgd*ead:,.0f}")
print("Use logistic loss to rank / train the PD model. Use ECL to talk capital and provisioning.")



In [ ]:
TRUE_W = np.array([-0.012, 8.5]); TRUE_B = 0.0
M_LIST = [40, 80, 160, 320, 640]
MISFILE_LIST = [0.00, 0.04, 0.08, 0.15, 0.25]
N_REPS = 18; SEED = 11

def make_book(m, misfile, seed):
    rng = np.random.default_rng(seed)
    cs = rng.normal(690, 55, m).clip(520, 820)
    dti_s = rng.normal(0.34, 0.11, m).clip(0.08, 0.75)
    X = np.column_stack([cs - 660.0, dti_s - 0.32])
    p = sigmoid(X @ TRUE_W + TRUE_B)
    y = (rng.random(m) < p).astype(float)
    nflip = int(misfile * m)
    if nflip:
        idx = rng.choice(m, size=nflip, replace=False)
        y[idx] = 1 - y[idx]
    return X, y

def sweep(values, kind):
    means, stds = [], []
    for v in values:
        js = []
        for r in range(N_REPS):
            if kind == "m":
                X, y = make_book(v, 0.04, SEED + 17*r + v)
            else:
                X, y = make_book(200, v, SEED + 31*r + int(100*v))
            js.append(compute_cost_logistic_vec(X, y, TRUE_W, TRUE_B))
        means.append(np.mean(js)); stds.append(np.std(js))
    return np.array(means), np.array(stds)

mm, ms = sweep(M_LIST, "m")
nm, ns = sweep(MISFILE_LIST, "n")
fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.5))
ax[0].errorbar(M_LIST, mm, yerr=ms, marker="o", capsize=3)
ax[0].set_xlabel("book size m"); ax[0].set_ylabel("J"); ax[0].set_title("J vs book size")
ax[0].grid(True, alpha=0.3)
ax[1].errorbar(MISFILE_LIST, nm, yerr=ns, marker="s", color="#d62728", capsize=3)
ax[1].set_xlabel("misfile rate"); ax[1].set_ylabel("J"); ax[1].set_title("J vs dirty outcomes")
ax[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("J by m", mm)
print("J by misfile", nm)



In [ ]:
credit_risk_note = """
S1 (b=−3) costs J≈0.367 against S2 at J≈0.504 on the same six files — S2 leaves a default on the wrong side of the cut.
Loop and vectorized BCE agree; clip PD to [1e-15, 1-1e-15]. Do not fit PD with squared error: the (w,b) heatmap is not a bowl.
J is a scoring rule, not a provision. Next control is ∇J and a threshold policy on a hold-out vintage.
"""
committee_note = """
Headline: on the six-file sketch the tighter underwriting cut is the better scoring rule (0.37 vs 0.50 average surprise).
A rule that is almost sure a loan will break, and then the customer pays, is treated as a large miss — which is what we want before rollout.
Ask model validation for the same comparison on a full vintage, plus a dollar ECL view for capital, before any policy change.
"""
branch_note = """
Think of a simple DTI rule: low DTI usually pays, high DTI more often breaks.
We keep the version of the rule that is less shocked when real customers pay or default.
This is not the collections number on the loan. It is only how well the rule’s ‘chance of breaking’ matches what happened.
"""
print(credit_risk_note); print(committee_note); print(branch_note)



In [ ]:
from IPython.display import Image, display
import os
for p in ["bank_logloss_cost_flowchart.png", "bank_logloss_curves.png",
          "bank_logloss_heatmap.png", "bank_logloss_boundaries.png",
          "bank_logloss_simulation.png", "bank_logloss_1d.png"]:
    if os.path.exists(p):
        display(Image(p, width=620))



## Key takeaways
1. Logistic loss scores a **PD**, not a dollar NPL.
2. MSE on PD is the wrong training / comparison objective.
3. Lower book $J$ means the current scorecard is more consistent with observed defaults.
4. Dirty outcome files inflate $J$ even when the scorecard is unchanged.
5. Quote $J$ to model risk; quote ECL to finance / capital.
